In [ ]:
import os
import joblib
from xgboost import XGBClassifier
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
DRIVE_PATH = "/content/drive/MyDrive/"

In [ ]:
rf_model = joblib.load(os.path.join(DRIVE_PATH, "Copy of random_forest_model.pkl"))

dt_model = joblib.load(os.path.join(DRIVE_PATH, "baseline_decision_tree_model.pkl"))

lgb_model = joblib.load(os.path.join(DRIVE_PATH, "Copy of lightgbm_model(1).pkl"))

label_encoder = joblib.load(os.path.join(DRIVE_PATH, "label_encoder.pkl"))

xgb_model = XGBClassifier()
xgb_model.load_model(os.path.join(DRIVE_PATH, "Copy of xgboost_model.json"))

print("✅ All models loaded successfully!")

/usr/local/lib/python3.12/dist-packages/sklearn/base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LabelEncoder from version 1.9.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


✅ All models loaded successfully!


In [ ]:
import pandas as pd

df = pd.read_csv("/content/drive/MyDrive/clean_190k_dataset.csv")

print(df.shape)

(189647, 378)


In [ ]:
print(df.columns)

Index(['diseases', 'anxiety and nervousness', 'depression',
       'shortness of breath', 'depressive or psychotic symptoms',
       'sharp chest pain', 'dizziness', 'insomnia',
       'abnormal involuntary movements', 'chest tightness',
       ...
       'stuttering or stammering', 'problems with orgasm', 'nose deformity',
       'lump over jaw', 'sore in nose', 'hip weakness', 'back swelling',
       'ankle stiffness or tightness', 'ankle weakness', 'neck weakness'],
      dtype='object', length=378)


In [ ]:
X = df.drop("diseases", axis=1)
y = df["diseases"]

print(X.shape)
print(y.shape)

(189647, 377)
(189647,)


In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("X_train:", X_train.shape)
print("X_test :", X_test.shape)
print("y_train:", y_train.shape)
print("y_test :", y_test.shape)

X_train: (151717, 377)
X_test : (37930, 377)
y_train: (151717,)
y_test : (37930,)


In [ ]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()

df['target'] = encoder.fit_transform(df['diseases'])

X = df.drop(['diseases', 'target'], axis=1)
y = df['target']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

print(X_train.shape)
print(X_test.shape)

(151717, 377)
(37930, 377)


In [ ]:
rf_pred = rf_model.predict(X_test)
dt_pred = dt_model.predict(X_test)
lgb_pred = lgb_model.predict(X_test)
xgb_pred = xgb_model.predict(X_test)

print("✅ All models predicted successfully!")

✅ All models predicted successfully!


In [ ]:
import numpy as np

# Get prediction probabilities
rf_prob = rf_model.predict_proba(X_test)
xgb_prob = xgb_model.predict_proba(X_test)
lgb_prob = lgb_model.predict_proba(X_test)

# Average probabilities (equal weights)
avg_prob = (rf_prob + xgb_prob + lgb_prob) / 3

# Final prediction
voting_pred1 = np.argmax(avg_prob, axis=1)

print("✅ Voting predictions created successfully!")

✅ Voting predictions created successfully!


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(y_test, voting_pred1)
precision = precision_score(y_test, voting_pred1, average='weighted', zero_division=0)
recall = recall_score(y_test, voting_pred1, average='weighted', zero_division=0)
f1 = f1_score(y_test, voting_pred1, average='weighted', zero_division=0)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

Accuracy : 0.8295
Precision: 0.8792
Recall   : 0.8295
F1 Score : 0.8476


In [ ]:
import joblib
import numpy as np

class SoftVotingClassifier:
    def __init__(self, rf_model, xgb_model, lgb_model):
        self.rf_model = rf_model
        self.xgb_model = xgb_model
        self.lgb_model = lgb_model

    def predict_proba(self, X):
        rf_prob = self.rf_model.predict_proba(X)
        xgb_prob = self.xgb_model.predict_proba(X)
        lgb_prob = self.lgb_model.predict_proba(X)

        return (rf_prob + xgb_prob + lgb_prob) / 3

    def predict(self, X):
        avg_prob = self.predict_proba(X)
        return np.argmax(avg_prob, axis=1)

In [ ]:
voting_model = SoftVotingClassifier(
    rf_model,
    xgb_model,
    lgb_model
)

print("✅ Voting model created!")

✅ Voting model created!


In [ ]:
import joblib

joblib.dump(voting_model, "/content/voting_classifier_rf_xgb_lgb.pkl")

['/content/voting_classifier_rf_xgb_lgb.pkl']

In [ ]:
import os

size = os.path.getsize("/content/voting_classifier_rf_xgb_lgb.pkl")
print(f"{size/(1024*1024):.2f} MB")

4164.58 MB


In [ ]:
from google.colab import files
files.download('/content/voting_classifier_rf_xgb_lgb.pkl')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!zip -0 /content/voting_classifier.zip /content/voting_classifier_rf_xgb_lgb.pkl

  adding: content/voting_classifier_rf_xgb_lgb.pkl


zip error: Interrupted (aborting)


In [ ]:
!cp /content/voting_classifier_rf_xgb_lgb.pkl /content/drive/MyDrive/

^C


In [ ]:
joblib.dump(
    voting_model,
    "/content/drive/MyDrive/voting_classifier_rf_xgb_lgb.pkl"
)

print("✅ Voting classifier saved successfully!")

✅ Voting classifier saved successfully!
